# Practical 1: From data to a model with the Keeling curve

**Intro to AI, Machine Learning and Data Science** · about 25 minutes

Since March 1958 the atmospheric CO$_2$ concentration has been measured on Mauna Loa, Hawaii. The resulting time series, the *Keeling curve*, is one of the most famous datasets in science. We will use it to walk through the full workflow from the lecture:

1. **Load and inspect** the data with pandas
2. **Find data-quality problems** before they find you
3. **Visualise** and summarise
4. **Build a model by hand**: linear regression trained with gradient descent in NumPy
5. **Test generalisation**: does the model predict years it has never seen?
6. *Bonus:* engineer better features and make a forecast

**How to work through this notebook**

* Run cells from top to bottom with `Shift + Enter`.
* Sections 1 to 5 are the core path: run through them first, and do the **Your turn** exercises if you have time. Section 6 is a bonus for fast finishers.
* **Your turn** cells are small exercises. A worked solution is hidden under *Show solution* just below each one. Try first, then peek.

*Data:* Dr. Xin Lan, NOAA/GML ([gml.noaa.gov/ccgg/trends](https://gml.noaa.gov/ccgg/trends/)) and Dr. Ralph Keeling, Scripps Institution of Oceanography ([scrippsco2.ucsd.edu](https://scrippsco2.ucsd.edu/)). Snapshot downloaded in September 2026.

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# A consistent, colour-blind-safe plotting style for the course
plt.rcParams.update({
    "figure.figsize": (9, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e1e0d9",
    "lines.linewidth": 2,
    "axes.prop_cycle": plt.cycler(color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
                                         "#e87ba4", "#008300", "#4a3aa7", "#e34948"]),
})

## 1. Load and inspect the data

The file is a CSV with a long comment header (lines starting with `#`). pandas can skip those for us.

In [ ]:
DATA = Path("data/co2_mm_mlo.csv")
URL = "https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.csv"

# Use the snapshot shipped with the course; fall back to NOAA's live file if it is missing
df = pd.read_csv(DATA if DATA.exists() else URL, comment="#")
df.head()

Each row is one month. The columns are:

| column | meaning |
|---|---|
| `year`, `month` | calendar month |
| `decimal date` | the middle of the month as a fractional year, e.g. `1958.2027` |
| `average` | monthly mean CO$_2$ mole fraction in ppm (parts per million) |
| `deseasonalized` | the same, with the average seasonal cycle removed |
| `ndays` | number of days with valid measurements that month |
| `sdev`, `unc` | standard deviation of the daily values, and uncertainty of the monthly mean |

`info()` tells us the size, the types and whether anything is missing. `describe()` gives summary statistics per column.

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

## 2. Data quality: look before you model

Look at the `min` row of `describe()` above. `ndays = -1`, `sdev = -9.99` and `unc = -0.99` are not measurements: a negative number of days is impossible. These are **sentinel values** meaning "no information". The file header explains that the Scripps record (1958 to 1974) has no daily statistics, and that NOAA's missing months were *interpolated*.

pandas did not complain, because `-9.99` is a perfectly valid float. This is the most common silent bug in data science: if you average `sdev` now, you get nonsense.

In [ ]:
no_info = df["ndays"] < 0
print(f"{no_info.sum()} of {len(df)} months have no daily information")
df.loc[no_info, "year"].value_counts().sort_index()

Let's replace the sentinels with `NaN` (pandas' "missing" marker), which every pandas function knows how to skip.

In [ ]:
clean = df.copy()
stats_cols = ["ndays", "sdev", "unc"]
clean[stats_cols] = clean[stats_cols].mask(clean[stats_cols] < 0)  # negative -> NaN
clean[stats_cols].isna().sum()

### Your turn

What is the mean of `sdev` in the raw table `df`, and in `clean`? Which one is physically meaningful?

In [ ]:
# Your code here

<details>
<summary>Show solution</summary>

```python
print("raw  :", df["sdev"].mean().round(3))
print("clean:", clean["sdev"].mean().round(3))
```

The raw mean is dragged down by the `-9.99` sentinels. Only the clean value (around 0.5 ppm, the typical day-to-day scatter within a month) means anything. Always check value ranges against what is physically possible.

</details>

> **Provenance matters too.** The file header notes that after the 2022 eruption of Mauna Loa, measurements from December 2022 to July 2023 came from a site about 21 miles away. Knowing how your data was produced is part of data science.

## 3. Visualise and summarise

In [ ]:
fig, ax = plt.subplots()
ax.plot(clean["decimal date"], clean["average"], lw=1, label="monthly mean")
ax.plot(clean["decimal date"], clean["deseasonalized"], label="seasonally adjusted")
ax.set(xlabel="year", ylabel="CO$_2$ (ppm)", title="Atmospheric CO$_2$ at Mauna Loa")
ax.legend();

Two things jump out: a **rising trend** that is slightly curved, and a regular **seasonal wiggle**. Is the rise speeding up? Let's compute the growth rate with a few lines of pandas: annual means (complete years only), year-on-year differences, then an average per decade.

In [ ]:
months_per_year = clean.groupby("year")["average"].count()
full_years = months_per_year[months_per_year == 12].index

annual = clean[clean["year"].isin(full_years)].groupby("year")["average"].mean()
growth = annual.diff().dropna()                         # ppm per year
by_decade = growth.groupby(growth.index // 10 * 10).mean()

ax = by_decade.plot.bar(color="#2a78d6", width=0.6, rot=0)
ax.set(xlabel="decade", ylabel="mean growth (ppm / year)", title="CO$_2$ growth rate by decade")
by_decade.round(2)

### Your turn

Which calendar month has the highest CO$_2$ on average, and which the lowest? *Hint:* `average - deseasonalized` isolates the seasonal cycle; group it by `month`.

In [ ]:
# Your code here

<details>
<summary>Show solution</summary>

```python
seasonal = (clean["average"] - clean["deseasonalized"]).groupby(clean["month"]).mean()
print(seasonal.round(2))
print("highest:", seasonal.idxmax(), " lowest:", seasonal.idxmin())
seasonal.plot.bar(rot=0, xlabel="month", ylabel="ppm relative to trend");
```

CO$_2$ peaks in May and bottoms out in September/October: the Northern Hemisphere's vegetation takes up CO$_2$ during its growing season. The data "breathes".

</details>

## 4. The machine-learning recipe, by hand

From the lecture, every ML method has four ingredients. For a straight-line fit of CO$_2$ against time:

| ingredient | here |
|---|---|
| **Data** | inputs $x_i$ = decimal date, targets $y_i$ = monthly CO$_2$ |
| **Model** | $\hat{y} = w\,x + b$ with parameters $w$ (slope) and $b$ (intercept) |
| **Loss** | mean squared error $L(w, b) = \frac{1}{n}\sum_i (\hat{y}_i - y_i)^2$ |
| **Optimiser** | gradient descent: $w \leftarrow w - \eta\,\partial L/\partial w$, same for $b$ |

The gradients of the loss are

$$\frac{\partial L}{\partial w} = \frac{2}{n}\sum_i (\hat{y}_i - y_i)\,x_i, \qquad \frac{\partial L}{\partial b} = \frac{2}{n}\sum_i (\hat{y}_i - y_i).$$

Yes, this problem has a closed-form solution (least squares). We use gradient descent anyway, because it is exactly how neural networks with millions of parameters are trained.

In [ ]:
x_raw = clean["decimal date"].to_numpy()
y = clean["average"].to_numpy()


def predict(x, w, b):
    return w * x + b


def mse(y_pred, y_true):
    return np.mean((y_pred - y_true) ** 2)


def gradient_descent(x, y, lr, n_steps=100):
    """Fit y = w*x + b by gradient descent. Returns w, b and the loss after each step."""
    w, b = 0.0, 0.0
    losses = []
    for step in range(n_steps):
        error = predict(x, w, b) - y
        dw = 2 * np.mean(error * x)   # dL/dw
        db = 2 * np.mean(error)       # dL/db
        w -= lr * dw                  # step downhill
        b -= lr * db
        losses.append(mse(predict(x, w, b), y))
    return w, b, np.array(losses)

### Attempt 1: raw inputs

Let's train on the dates as they are (values around 1958 to 2026).

In [ ]:
for lr in [1e-7, 3e-7]:
    w, b, losses = gradient_descent(x_raw, y, lr=lr)
    print(f"lr = {lr:g}:  final loss = {losses[-1]:.3g},  w = {w:.4f},  b = {b:.4f}")

With `lr = 3e-7` the loss explodes. With `lr = 1e-7` it is stable but the intercept $b$ has barely moved from zero, so the "fit" is a line forced through the origin. The problem is **scale**: $x \approx 2000$ makes the loss surface extremely stretched in the $w$ direction (lecture slide on feature scaling), so any learning rate small enough to be stable in $w$ is far too small for $b$.

### Attempt 2: standardised inputs

The fix is to **standardise** the input: subtract the mean, divide by the standard deviation, so $x$ is of order 1.

In [ ]:
x_mean, x_std = x_raw.mean(), x_raw.std()
x = (x_raw - x_mean) / x_std

fig, ax = plt.subplots()
for lr in [0.001, 0.01, 0.1, 1.05]:
    w, b, losses = gradient_descent(x, y, lr=lr)
    ax.plot(losses, label=f"lr = {lr}")
    print(f"lr = {lr:<6} final loss = {losses[-1]:.4g}")
ax.set(yscale="log", xlabel="gradient descent step", ylabel="loss (MSE, ppm$^2$)",
       title="Training curves for four learning rates")
ax.legend();

Read the plot like a practitioner:

* `0.001`: stable, but hopelessly slow.
* `0.01`: getting there, not converged after 100 steps.
* `0.1`: converges quickly to the minimum.
* `1.05`: each step overshoots further; the loss **diverges**.

Choosing the learning rate is the first hyperparameter everyone tunes, and you will meet it again whenever you train a neural network.

Now train with a good learning rate and compare against NumPy's closed-form least-squares fit.

In [ ]:
w, b, losses = gradient_descent(x, y, lr=0.1)
w_ls, b_ls = np.polyfit(x, y, deg=1)   # closed-form least squares
print(f"gradient descent: w = {w:.4f}, b = {b:.4f}")
print(f"least squares   : w = {w_ls:.4f}, b = {b_ls:.4f}")

fig, ax = plt.subplots()
ax.plot(x_raw, y, lw=1, label="data")
ax.plot(x_raw, predict(x, w, b), label="linear model")
ax.set(xlabel="year", ylabel="CO$_2$ (ppm)", title="A straight line underfits the Keeling curve")
ax.legend();

### Your turn

The slope `w` is in "ppm per standard deviation of time". Convert it back to **ppm per year**. Compare it with the decade growth rates you computed in section 3.

In [ ]:
# Your code here

<details>
<summary>Show solution</summary>

```python
print(f"{w / x_std:.2f} ppm per year")
```

Because $x = (t - \bar t)/s$, the model $w x + b$ has slope $w/s$ per year: about 1.6 ppm/year. That is a single average rate, while section 3 showed the rate climbing from under 1 to over 2.5 ppm/year. The straight line is too simple for this data: it **underfits**.

</details>

## 5. Generalisation: can the model predict the future?

A model is only useful if it works on data it has not seen. For a time series the honest test is to **train on the past and test on the future**. We keep everything up to the end of 2005 for training and hold out 2006 onwards as a test set.

Notice that we compute the scaling statistics from the **training set only**. Using the test years to compute the mean and standard deviation would leak information from the future into training.

In [ ]:
is_train = clean["decimal date"] < 2006
t_train, y_train = x_raw[is_train], y[is_train]
t_test, y_test = x_raw[~is_train], y[~is_train]

mu, sigma = t_train.mean(), t_train.std()      # training statistics only!
s_train, s_test = (t_train - mu) / sigma, (t_test - mu) / sigma
print(f"{len(t_train)} training months, {len(t_test)} test months")


def rmse(y_pred, y_true):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))

Now fit polynomials of increasing degree (more flexible models) and compare training and test error.

In [ ]:
rows = []
fig, ax = plt.subplots()
ax.plot(x_raw, y, color="#898781", lw=1, label="data")
ax.axvline(2006, color="#52514e", lw=1)
ax.text(2006.5, 318, "test period", color="#52514e")
for degree in [1, 2, 4, 12]:
    coeffs = np.polyfit(s_train, y_train, deg=degree)
    rows.append({
        "degree": degree,
        "train RMSE (ppm)": rmse(np.polyval(coeffs, s_train), y_train),
        "test RMSE (ppm)": rmse(np.polyval(coeffs, s_test), y_test),
    })
    ax.plot(x_raw, np.polyval(coeffs, (x_raw - mu) / sigma), label=f"degree {degree}")
ax.set(ylim=(300, 460), xlabel="year", ylabel="CO$_2$ (ppm)", title="Polynomial trends fitted on 1958 to 2005")
ax.legend(ncol=5, loc="upper left")
pd.DataFrame(rows).round(2)

This table is the whole story of the lecture's over/underfitting slide:

* **Training error only goes down** as the model gets more flexible. It cannot tell you which model is best.
* **Test error** is what matters. Degree 1 underfits. Degree 2 captures the accelerating trend. Degree 12 has the lowest training error and a catastrophic test error: it **overfits** and **extrapolates wildly**.

ML models are generally good at *interpolation* and unreliable at *extrapolation*. Be very careful whenever you predict outside the range of your training data.

### Your turn

Repeat the comparison with a **random** split (80 % of months for training, 20 % for testing, chosen at random). What happens to the test error of the degree-12 model? Why is this result misleading if the goal is forecasting?

In [ ]:
# Your code here

<details>
<summary>Show solution</summary>

```python
rng = np.random.default_rng(0)
is_train_random = rng.random(len(x_raw)) < 0.8
mu_r, sigma_r = x_raw[is_train_random].mean(), x_raw[is_train_random].std()
s_all = (x_raw - mu_r) / sigma_r
for degree in [1, 2, 4, 12]:
    coeffs = np.polyfit(s_all[is_train_random], y[is_train_random], deg=degree)
    test_err = rmse(np.polyval(coeffs, s_all[~is_train_random]), y[~is_train_random])
    print(f"degree {degree:>2}: random-split test RMSE = {test_err:.2f} ppm")
```

With a random split every test month sits between training months, so the model only has to **interpolate**. Degree 12 now looks as good as degree 2, but it would still fail at forecasting. The split must mimic how the model will be used. For forecasting that means splitting by time; for other problems it may mean splitting by user, by machine, by experiment or by site. We come back to this in Part 2 under *data leakage*.

</details>

## 6. Bonus: better features beat a bigger model

Instead of a high-degree polynomial, use what we *know* about the system: a smooth, slowly accelerating trend plus a yearly cycle. We build those as input **features** and fit a linear model on them by least squares:

$$\hat{y} = \beta_0 + \beta_1 s + \beta_2 s^2 + \beta_3 \sin(2\pi t) + \beta_4 \cos(2\pi t) + \beta_5 \sin(4\pi t) + \beta_6 \cos(4\pi t)$$

The model is still *linear in its parameters* $\beta$, even though it is curved in time. Feature engineering like this is how what you already know about a problem enters classical ML.

In [ ]:
def features(t):
    s = (t - mu) / sigma
    return np.column_stack([
        np.ones_like(t), s, s**2,
        np.sin(2 * np.pi * t), np.cos(2 * np.pi * t),   # yearly cycle
        np.sin(4 * np.pi * t), np.cos(4 * np.pi * t),   # its first harmonic
    ])


beta, *_ = np.linalg.lstsq(features(t_train), y_train, rcond=None)
print(f"train RMSE = {rmse(features(t_train) @ beta, y_train):.2f} ppm")
print(f"test  RMSE = {rmse(features(t_test) @ beta, y_test):.2f} ppm")

t_future = np.arange(1958, 2036, 1 / 12)
fig, ax = plt.subplots()
ax.plot(x_raw, y, color="#898781", lw=1, label="data")
ax.plot(t_future, features(t_future) @ beta, lw=1.2, label="trend + seasonal model (trained on 1958 to 2005)")
ax.axvline(2006, color="#52514e", lw=1)
ax.set(xlabel="year", ylabel="CO$_2$ (ppm)")
ax.legend(loc="upper left");

### Your turn

Refit the same model on **all** the data, then use it to estimate when the monthly mean first exceeds 450 ppm. How much would you trust this forecast, and why?

In [ ]:
# Your code here

<details>
<summary>Show solution</summary>

```python
beta_all, *_ = np.linalg.lstsq(features(x_raw), y, rcond=None)
t_future = np.arange(2026, 2050, 1 / 12)
forecast = features(t_future) @ beta_all
first = t_future[np.argmax(forecast > 450)]
print(f"first month above 450 ppm: about {first:.1f}")
```

The model says the early-to-mid 2030s. Trust it only as "if the past trend continues": the model knows nothing about emissions policy, the carbon cycle or El Niño. It is a statistical extrapolation, not a physical projection.

</details>

## Summary

* **Look at the data first.** Sentinel values, gaps and changes of instrument are everywhere in real measurements.
* **Every ML model is data + model + loss + optimiser.** You just implemented all four in NumPy.
* **Scale your inputs** so that gradient descent behaves.
* **The learning rate** controls speed and stability.
* **Judge a model on held-out data**, split in the way the model will be used (here: by time).
* **Prior knowledge built into the features** often beats a more complex model.

Next: in Part 2 we let scikit-learn do the heavy lifting.